In [49]:
from datetime import datetime
import pandas as pd
import yfinance as yf
import requests
import zipfile
from io import BytesIO
import numpy as np

In [50]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# yhfiance에서 Brent 가격 뽑아오기

1. Brent 가격 (BZ=f)

- 국제 기준 원유 가격 그 자체임: 브랜트유 선물 종가를 기준으로 한 국제 유가
- 대부분의 유가 뉴스나 리포트에서 '국제유가 상승'이라 함은 얘 말함

2. WTI & Brent–WTI 스프레드

- '지역 간 가격차'로 수급 구조를 반영하는 보조 피처
- WTI는 미국 원유 선물인데, 미국 내 생산/재고/수출입 이슈에 더 민감함
- 스프레드 = brent 종가 - wti 종가

3. 변수 설명

- close: 종가
- ret_n: n일 로그 수익률 - 단기/중기 모멘텀 및 변동성 피처로 활용
- ma_n: n일 이동평균선 (예를들어서 5일짜리>20일짜리: 단기 강세 모멘텀)
- vol_5d: 최근 5일간 수익률 표준편차. 얘도 단기 리스크 척도
- high_low_range: 당일 고-저 비율. 변동성이나 시장 불안도 지표

스프레드가 늘어나면 그니까 브렌트가 더 비싸면 -> 유럽/아시아 수급이 타이트

스프테드가 줄면 -> 미국산 원유 공급 증가 or 글로벌 수요 약화


**브렌트 자체가 유가의 절대수준**

**스프레드는 지역 간 수급 불균형 지표**

라서 경제적 맥락 읽힐라면 필요하다

In [51]:
def make_brent_wti_features(start="2013-09-01", end=None):
    brent = yf.download("BZ=F", start=start, end=end, auto_adjust=False, progress=False)
    wti   = yf.download("CL=F", start=start, end=end, auto_adjust=False, progress=False)

    brent = brent.rename(columns=str.lower)
    wti   = wti.rename(columns=str.lower)

    df = pd.DataFrame(index=brent.index)
    df["brent_close"] = brent["close"]
    df["wti_close"] = wti["close"].reindex(df.index)

    # 스프레드
    df["brent_wti_spread"] = df["brent_close"] - df["wti_close"]

    # 수익률
    df["brent_ret_1d"] = df["brent_close"].pct_change(1)
    df["brent_ret_5d"] = df["brent_close"].pct_change(5)
    df["brent_ret_20d"] = df["brent_close"].pct_change(20)

    # 이동평균
    df["brent_ma_5"] = df["brent_close"].rolling(5).mean()
    df["brent_ma_20"] = df["brent_close"].rolling(20).mean()
    df["brent_ma_60"] = df["brent_close"].rolling(60).mean()

    # 변동성 proxy
    df["brent_vol_5d"] = df["brent_close"].pct_change().rolling(5).std()
    df["high_low_range"] = (brent["high"] - brent["low"]) / brent["close"]

    return df.dropna()

features = make_brent_wti_features()
features = features[features.index >= "2014-01-01"]
print(features.tail())


            brent_close  wti_close  brent_wti_spread  brent_ret_1d  \
Date                                                                 
2025-11-10    64.059998  60.130001          3.929996      0.006758   
2025-11-11    65.160004  61.040001          4.120003      0.017171   
2025-11-12    62.709999  58.490002          4.219997     -0.037600   
2025-11-13    63.009998  58.689999          4.320000      0.004784   
2025-11-14    64.389999  60.090000          4.299999      0.021901   

            brent_ret_5d  brent_ret_20d  brent_ma_5  brent_ma_20  brent_ma_60  \
Date                                                                            
2025-11-10     -0.012791       0.011687   63.806001      63.6215    65.813667   
2025-11-11      0.011173       0.044398   63.950001      63.7600    65.789667   
2025-11-12     -0.012752       0.012922   63.788000      63.8000    65.738333   
2025-11-13     -0.005838       0.031936   63.714000      63.8975    65.674500   
2025-11-14      0.01194

In [52]:
features = features.reset_index()
features.head()

,Date,brent_close,wti_close,brent_wti_spread,brent_ret_1d,brent_ret_5d,brent_ret_20d,brent_ma_5,brent_ma_20,brent_ma_60,brent_vol_5d,high_low_range
0,2014-01-02,107.779999,95.440002,12.339996,-0.027256,-0.036819,-0.032930,110.790001,110.6705,109.391334,0.011831,0.033680
1,2014-01-03,106.889999,93.959999,12.930000,-0.008258,-0.045455,-0.050879,109.772000,110.3840,109.356167,0.010935,0.017214
2,2014-01-06,106.730003,93.430000,13.300003,-0.001497,-0.048583,-0.046031,108.682001,110.1265,109.310667,0.010182,0.012836
3,2014-01-07,107.349998,93.669998,13.680000,0.005809,-0.034709,-0.032709,107.910001,109.9450,109.271834,0.012422,0.007452
4,2014-01-08,107.150002,92.330002,14.820000,-0.001863,-0.032942,-0.039961,107.180000,109.7220,109.221667,0.012569,0.008493


# EIA 데이터에서 수급 & 재고 관련한거 가져오기


### 상업 원유/휘발유/증류유 재고

1. crude_stock_level(원유 재고)

- 미국 상업용 원유 재고 총량
- 재고 늘면 > 공급 과잉 > 유가 하방 압력 요런식

2. gas_stock_level (휘발유 재고)

- 정제제품(가솔린) 재고.
- 운송 수요랑 계절 요인(여름 드라이빙 시즌) 반영함
- 재고 늘면 수요 둔화 가능성

3. dist_stock_level (증류유 재고)

- 경유랑 항공유 등 산업용 연료 재고

4. _wow_change (주간 변화량)

- 전 주 대비 재고 증감 > 단기 공급 충격 포착 ㄱㄴ
- 재고 급감 > 공급 차질 우려 > 단기 유가 상승

5. _vs_5yr_avg (5년 평균 대비 편차)

- 계절적 요인 제거할라고
- 같은 시기 과거 5년 평균 대비 얼마나 높은지 낮은지로 구조적 과잉/부족 상황 판단.

In [53]:
# API_KEY = "m9fePZTks6kdpVDleRYzWYJMdZOXubKwwTHYuiPJ"

# # ---------- 1) Crude: SAX (Ending Stocks Excluding SPR) ----------
# url_crude = "https://api.eia.gov/v2/petroleum/stoc/wstk/data/"

# params_crude = {
#     "api_key": API_KEY,
#     "frequency": "weekly",
#     "data[0]": "value",
#     "facets[process][]": "SAX",   # Ending Stocks Excluding SPR
#     "start": "2014-01-01",
#     "end": "2025-10-31",
#     "sort[0][column]": "period",
#     "sort[0][direction]": "asc",
#     "offset": 0,
#     "length": 5000,
# }

# res = requests.get(url_crude, params=params_crude)
# res.raise_for_status()
# j = res.json()

# df_crude = pd.DataFrame(j["response"]["data"])
# df_crude["date"] = pd.to_datetime(df_crude["period"])
# df_crude = df_crude.sort_values("date")

# # 미국 전체 + Crude Oil + Ending Stocks Excluding SPR
# mask_crude = (
#     (df_crude["area-name"] == "U.S.") &
#     (df_crude["product-name"].str.contains("Crude Oil", case=False)) &
#     (df_crude["process-name"] == "Ending Stocks Excluding SPR")
# )

# crude = (df_crude[mask_crude][["date", "value"]]
#          .drop_duplicates(subset="date", keep="last")
#          .set_index("date")
#          .sort_index()
#          .rename(columns={"value": "crude_stock_level"}))

# crude["crude_stock_level"] = pd.to_numeric(crude["crude_stock_level"], errors="coerce")
# print("crude:", crude.shape, crude.index.min(), crude.index.max())


In [54]:
API_KEY = "m9fePZTks6kdpVDleRYzWYJMdZOXubKwwTHYuiPJ"

def fetch_crude_stock():
    url = "https://api.eia.gov/v2/seriesid/PET.WCESTUS1.W"
    params = {"api_key": API_KEY}
    r = requests.get(url, params=params)
    r.raise_for_status()
    js = r.json()

    df = pd.DataFrame(js["response"]["data"])
    df["period"] = pd.to_datetime(df["period"])
    df["value"] = pd.to_numeric(df["value"])
    df = df.sort_values("period")

    df = df.rename(columns={"value": "crude_stock_level"})
    return df[["period", "crude_stock_level"]]

crude = fetch_crude_stock()
print(crude.head())
print(crude.tail())


         period  crude_stock_level
2249 1982-08-20             338764
2248 1982-08-27             336138
2247 1982-09-24             335586
2246 1982-10-01             334786
2245 1982-10-08             335260
      period  crude_stock_level
4 2025-10-10             423785
3 2025-10-17             422824
2 2025-10-24             415966
1 2025-10-31             421168
0 2025-11-07             427581


In [55]:
BASE = "https://api.eia.gov/v2/petroleum/stoc/wstk/data/"

def fetch_raw(product_code: str):
    params = {
        "api_key": API_KEY,
        "frequency": "weekly",
        "data[0]": "value",
        "facets[product][]": product_code,
        "start": "2009-01-01",
        "end": "2025-10-31",
        "sort[0][column]": "period",
        "sort[0][direction]": "asc",
        "offset": 0,
        "length": 5000,
    }
    r = requests.get(BASE, params=params)
    r.raise_for_status()
    data = r.json()["response"]["data"]
    df = pd.DataFrame(data)
    df["date"] = pd.to_datetime(df["period"])
    df = df.sort_values("date")
    return df

def extract_series(df, *, area="U.S.", product_kw=None, process_kw=None, col_name="value"):
    m = (df["area-name"] == area)

    if product_kw is not None:
        m &= df["product-name"].str.contains(product_kw, case=False)

    if process_kw is not None:
        m &= df["process-name"].str.contains(process_kw, case=False)

    out = (df[m][["date", "value"]]
           .drop_duplicates(subset="date", keep="last")
           .set_index("date")
           .sort_index()
           .rename(columns={"value": col_name}))
    out[col_name] = pd.to_numeric(out[col_name], errors="coerce")
    return out


In [56]:
def fetch_gas_stock_total():
    url = "https://api.eia.gov/v2/seriesid/PET.WGTSTUS1.W"
    params = {"api_key": API_KEY}
    r = requests.get(url, params=params)
    r.raise_for_status()
    js = r.json()

    df = pd.DataFrame(js["response"]["data"])
    df["period"] = pd.to_datetime(df["period"])
    df["value"] = pd.to_numeric(df["value"])
    df = df.sort_values("period")

    df = df.rename(columns={"value": "gas_stock_level"})
    return df[["period", "gas_stock_level"]]


gas = fetch_gas_stock_total()

print("gas:", gas.shape, gas.index.min(), gas.index.max())


gas: (1871, 2) 0 1870


In [57]:
gas.head()

,period,gas_stock_level
1870,1990-01-05,210982
1869,1990-01-12,215395
1868,1990-01-19,219221
1867,1990-01-26,229695
1866,1990-02-02,234249


In [58]:
def fetch_dist_stock_total():
    url = "https://api.eia.gov/v2/seriesid/PET.WGTSTUS1.W"
    params = {"api_key": API_KEY}
    r = requests.get(url, params=params)
    r.raise_for_status()
    js = r.json()

    df = pd.DataFrame(js["response"]["data"])
    df["period"] = pd.to_datetime(df["period"])
    df["value"] = pd.to_numeric(df["value"])
    df = df.sort_values("period")

    df = df.rename(columns={"value": "dist_stock_level"})
    return df[["period", "dist_stock_level"]]

dist = fetch_dist_stock_total()

print("dist:", dist.shape, dist.index.min(), dist.index.max())


dist: (1871, 2) 0 1870


In [59]:
# crude, gas, dist 각각 이런 형태라고 가정:
# crude: [period, crude_stock_level]
# gas:   [period, gas_stock_level]
# dist:  [period, dist_stock_level]

def set_period_index(df, value_col):
    out = df.copy()
    out["period"] = pd.to_datetime(out["period"])
    out = out.set_index("period").sort_index()
    return out[[value_col]]

crude_i = set_period_index(crude, "crude_stock_level")
gas_i   = set_period_index(gas, "gas_stock_level")
dist_i  = set_period_index(dist, "dist_stock_level")

# 이제 같은 날짜 기준으로 옆으로 붙이기
stocks = pd.concat([crude_i, gas_i, dist_i], axis=1).sort_index()

print(stocks.tail())
print(stocks.dtypes)


            crude_stock_level  gas_stock_level  dist_stock_level
period                                                          
2025-10-10             423785         218826.0          218826.0
2025-10-17             422824         216679.0          216679.0
2025-10-24             415966         210738.0          210738.0
2025-10-31             421168         206009.0          206009.0
2025-11-07             427581         205064.0          205064.0
crude_stock_level      int64
gas_stock_level      float64
dist_stock_level     float64
dtype: object


In [60]:
stocks = stocks.reset_index()

stocks.head()

,period,crude_stock_level,gas_stock_level,dist_stock_level
0,1982-08-20,338764,NaN,NaN
1,1982-08-27,336138,NaN,NaN
2,1982-09-24,335586,NaN,NaN
3,1982-10-01,334786,NaN,NaN
4,1982-10-08,335260,NaN,NaN


### 파생변수들

In [61]:
# WoW
stocks["crude_stock_wow_change"] = stocks["crude_stock_level"].diff()
stocks["gas_stock_wow_change"] = stocks["gas_stock_level"].diff()
stocks["dist_stock_wow_change"] = stocks["dist_stock_level"].diff()

# 5년 평균 대비 (%)
# 주간 데이터이므로 1년 ≈ 52주 → 5년 = 260주
window_5yr = 52 * 5

stocks["crude_stock_vs_5yr_avg"] = (
    stocks["crude_stock_level"] / stocks["crude_stock_level"].rolling(window_5yr, min_periods=52).mean() - 1
)

stocks["gas_stock_vs_5yr_avg"] = (
    stocks["gas_stock_level"] / stocks["gas_stock_level"].rolling(window_5yr, min_periods=52).mean() - 1
)

stocks["dist_stock_vs_5yr_avg"] = (
    stocks["dist_stock_level"] / stocks["dist_stock_level"].rolling(window_5yr, min_periods=52).mean() - 1
)
stocks = stocks[stocks['period'] >= "2014-01-01"]

stocks.head()

,period,crude_stock_level,gas_stock_level,dist_stock_level,crude_stock_wow_change,gas_stock_wow_change,dist_stock_wow_change,crude_stock_vs_5yr_avg,gas_stock_vs_5yr_avg,dist_stock_vs_5yr_avg
1631,2014-01-03,326737,226959.0,226959.0,-3958.0,6243.0,6243.0,-0.025439,0.048841,0.048841
1632,2014-01-10,319079,233142.0,233142.0,-7658.0,6183.0,6183.0,-0.048331,0.077163,0.077163
1633,2014-01-17,320069,235265.0,235265.0,990.0,2123.0,2123.0,-0.045371,0.086674,0.086674
1634,2014-01-24,326490,234446.0,234446.0,6421.0,-819.0,-819.0,-0.026205,0.082617,0.082617
1635,2014-01-31,326930,234951.0,234951.0,440.0,505.0,505.0,-0.024830,0.084614,0.084614


In [62]:
# 1. 타입 통일
features["Date"] = pd.to_datetime(features["Date"])
stocks["period"] = pd.to_datetime(stocks["period"])

# 2. 인덱스 설정
features = features.set_index("Date").sort_index()
stocks = stocks.set_index("period").sort_index()

# 3. 인덱스 기준 left join
merged = features.merge(stocks, how="left", left_index=True, right_index=True)

# 4. 주간 값 forward-fill
merged = merged.ffill()

# 5. 인덱스를 Date 컬럼으로 복원
merged = merged.reset_index()  # 여기서 이미 'Date'라는 이름으로 나옴 (왼쪽 인덱스)
merged = merged.sort_values("Date")

print(merged.head())
print(merged.columns)

        Date  brent_close  wti_close  brent_wti_spread  brent_ret_1d  \
0 2014-01-02   107.779999  95.440002         12.339996     -0.027256   
1 2014-01-03   106.889999  93.959999         12.930000     -0.008258   
2 2014-01-06   106.730003  93.430000         13.300003     -0.001497   
3 2014-01-07   107.349998  93.669998         13.680000      0.005809   
4 2014-01-08   107.150002  92.330002         14.820000     -0.001863   

   brent_ret_5d  brent_ret_20d  brent_ma_5  brent_ma_20  brent_ma_60  \
0     -0.036819      -0.032930  110.790001     110.6705   109.391334   
1     -0.045455      -0.050879  109.772000     110.3840   109.356167   
2     -0.048583      -0.046031  108.682001     110.1265   109.310667   
3     -0.034709      -0.032709  107.910001     109.9450   109.271834   
4     -0.032942      -0.039961  107.180000     109.7220   109.221667   

   brent_vol_5d  high_low_range  crude_stock_level  gas_stock_level  \
0      0.011831        0.033680                NaN             

## 미국 원유 생산량

1. prod_weekly: 보통 1000베럴/데이 기준
- 공급측 기본체력 지표
- 추세적으로 증가하면 생산 여력 확대 > 공급 완화 > 유가 상방 압력 완화
- 단기 급락? 그때 이슈들을 봐야함

2. prod_4w_ma: 최근 4주 이동평균 생산량
- 주간 노이즈(점검 셧다운 등)를 줄이고 생산 추세 부ㄹ드럽게 본거
- 현재 값 vs 추세 갭 분석 하는 용도

3. prod_wow_change: 주간 생산량 변화율
- 생산이 얼마나 빠르게 늘고 주는지 보는거
- 단기간 커짐 > 공급 증가 기대 > 유가 하방 압력 요인


## 원유 수입/수출 & 순수입

1. crude_imports: 의미 주간 원유 수입량
- 미국 내 수요/정제수요 + 해외 공급 여건 반영
- 수입 감소 > 글로벌 공급 타이트/지정학 리스크/생산 증가 중 일부 가능성 ㅇㅇ
- 수요 빼매 들어오는지? 공급 여유가 있어서 싸게 오는지? 구분하는거

2. crude_experts: 미국 주간 원유 수출량
- 글로벌 타이트한 수급 > 미국산 경쟁력 높고 > 국제 유가 상방 환경일수도
- 글로벌 유동성 보는 피처

3. import_4w_ma: 원유 수입량 4주 이동평균

4. net_imports: 순수입=수출 - 수입
- 미국 내부 기준으론 공급 여력↑, 하지만 글로벌 시장으론 미국발 공급 확대 가능 → 국제 유가와의 관계는 복합적.

## 정유 설비 가동률

1. refinery_run_rate: 미국 정유공장의 가동률(퍼센트)

- 최종 수요에 가장 근접한 실물지표 중 하나
- 가동률 커지면 정제 제품 수요 견조/성수기 진입 > 원유 수요 증가 > 유가 상방 압력
- 시즌성(여름 드라이빙, 겨울 난방 등) 반영하는 피처, 재고 수입 가격이랑 같이 "수요 강도 vs 공급 여유"해석하는데 중요


In [63]:
# 원유 생산량
API_KEY = "m9fePZTks6kdpVDleRYzWYJMdZOXubKwwTHYuiPJ"

url = "https://api.eia.gov/v2/petroleum/sum/sndw/data/"
params = {
    "api_key": API_KEY,
    "frequency": "weekly",
    "data[0]": "value",
    "facets[series][]": "WCRFPUS2",
    "sort[0][column]": "period",
    "sort[0][direction]": "asc",
    "offset": 0,
    "length": 5000,
}

resp = requests.get(url, params=params)
resp.raise_for_status()
data = resp.json()["response"]["data"]
df = pd.DataFrame(data)

# 기본 정리
df["period"] = pd.to_datetime(df["period"])
df["value"] = pd.to_numeric(df["value"])
df = df.sort_values("period")



# 피처 생성
df["prod_weekly"] = df["value"]
df["prod_4w_ma"] = df["prod_weekly"].rolling(window=4).mean()
df["prod_wow_change"] = df["prod_weekly"].pct_change()

df = df[df["period"] >= "2014-01-01"].reset_index(drop=True)

# 필요 컬럼만
feat = df[["period", "prod_weekly", "prod_4w_ma", "prod_wow_change"]]
print(feat.tail())

        period  prod_weekly  prod_4w_ma  prod_wow_change
614 2025-10-10        13636    13567.75         0.000514
615 2025-10-17        13629    13599.75        -0.000513
616 2025-10-24        13644    13634.50         0.001101
617 2025-10-31        13651    13640.00         0.000513
618 2025-11-07        13862    13696.50         0.015457


In [64]:
feat.head()

,period,prod_weekly,prod_4w_ma,prod_wow_change
0,2014-01-03,8145,8108.75,0.002955
1,2014-01-10,8159,8134.00,0.001719
2,2014-01-17,8052,8119.25,-0.013114
3,2014-01-24,8044,8100.00,-0.000994
4,2014-01-31,8044,8074.75,0.000000


In [65]:
feat["period"] = pd.to_datetime(feat["period"])
feat = feat.set_index("period").sort_index()

# merged도 datetime 정렬 확인
merged["Date"] = pd.to_datetime(merged["Date"])
merged = merged.set_index("Date").sort_index()

# 병합 (왼쪽 = merged, 오른쪽 = feat)
merged = (
    merged
    .merge(feat, how="left", left_index=True, right_index=True)
    .ffill()  # 생산량이 주간이므로 주중엔 그대로 유지
    .reset_index()
    .rename(columns={"index": "Date"})
    .sort_values("Date")
)
merged.head()

C:\Users\juhye\AppData\Local\Temp\ipykernel_27616\1401098738.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  feat["period"] = pd.to_datetime(feat["period"])


,Date,brent_close,wti_close,brent_wti_spread,brent_ret_1d,brent_ret_5d,brent_ret_20d,brent_ma_5,brent_ma_20,brent_ma_60,brent_vol_5d,high_low_range,crude_stock_level,gas_stock_level,dist_stock_level,crude_stock_wow_change,gas_stock_wow_change,dist_stock_wow_change,crude_stock_vs_5yr_avg,gas_stock_vs_5yr_avg,dist_stock_vs_5yr_avg,prod_weekly,prod_4w_ma,prod_wow_change
0,2014-01-02,107.779999,95.440002,12.339996,-0.027256,-0.036819,-0.032930,110.790001,110.6705,109.391334,0.011831,0.033680,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2014-01-03,106.889999,93.959999,12.930000,-0.008258,-0.045455,-0.050879,109.772000,110.3840,109.356167,0.010935,0.017214,326737.0,226959.0,226959.0,-3958.0,6243.0,6243.0,-0.025439,0.048841,0.048841,8145.0,8108.75,0.002955
2,2014-01-06,106.730003,93.430000,13.300003,-0.001497,-0.048583,-0.046031,108.682001,110.1265,109.310667,0.010182,0.012836,326737.0,226959.0,226959.0,-3958.0,6243.0,6243.0,-0.025439,0.048841,0.048841,8145.0,8108.75,0.002955
3,2014-01-07,107.349998,93.669998,13.680000,0.005809,-0.034709,-0.032709,107.910001,109.9450,109.271834,0.012422,0.007452,326737.0,226959.0,226959.0,-3958.0,6243.0,6243.0,-0.025439,0.048841,0.048841,8145.0,8108.75,0.002955
4,2014-01-08,107.150002,92.330002,14.820000,-0.001863,-0.032942,-0.039961,107.180000,109.7220,109.221667,0.012569,0.008493,326737.0,226959.0,226959.0,-3958.0,6243.0,6243.0,-0.025439,0.048841,0.048841,8145.0,8108.75,0.002955


In [66]:
# 수입/수출

API_KEY = "m9fePZTks6kdpVDleRYzWYJMdZOXubKwwTHYuiPJ"

url = "https://api.eia.gov/v2/petroleum/move/wkly/data/"

params = {
    "api_key": API_KEY,
    "frequency": "weekly",
    "data[0]": "value",
    # Crude Oil Imports / Exports
    "facets[series][]": ["WCRIMUS2", "WCREXUS2"],
    "sort[0][column]": "period",
    "sort[0][direction]": "asc",
    "offset": 0,
    "length": 5000,
}

resp = requests.get(url, params=params)
resp.raise_for_status()
data = resp.json()["response"]["data"]
df = pd.DataFrame(data)

# 기본 정리
df["period"] = pd.to_datetime(df["period"])
df["value"] = pd.to_numeric(df["value"])
df = df.sort_values("period")

# Wide 형태로 피벗 (series별 컬럼 생성)
wide = df.pivot(index="period", columns="series", values="value").sort_index()

# 이름 정리
wide = wide.rename(columns={
    "WCRIMUS2": "crude_imports",
    "WCREXUS2": "crude_exports",
})

# 파생 피처
wide["import_4w_ma"] = wide["crude_imports"].rolling(4).mean()
wide["net_imports"] = wide["crude_imports"] - wide["crude_exports"]


wide = wide[wide.index >= "2014-01-01"]
wide.columns.name = None
wide = wide.reset_index()

print(wide.tail())


        period  crude_exports  crude_imports  import_4w_ma  net_imports
614 2025-10-10         4466.0         5525.0       6064.00       1059.0
615 2025-10-17         4203.0         5918.0       5919.75       1715.0
616 2025-10-24         4361.0         5051.0       5724.25        690.0
617 2025-10-31         4367.0         5924.0       5604.50       1557.0
618 2025-11-07         2816.0         5222.0       5528.75       2406.0


In [67]:
# 병합 전 형식 통일
wide["period"] = pd.to_datetime(wide["period"])
merged["Date"] = pd.to_datetime(merged["Date"])

# 인덱스 설정
wide = wide.set_index("period").sort_index()
merged = merged.set_index("Date").sort_index()

# left join + forward fill
merged = (
    merged
    .merge(wide, how="left", left_index=True, right_index=True)
    .ffill()  # 주간 데이터 보간
    .reset_index()
    .rename(columns={"index": "Date"})
    .sort_values("Date")
)

merged.head()

,Date,brent_close,wti_close,brent_wti_spread,brent_ret_1d,brent_ret_5d,brent_ret_20d,brent_ma_5,brent_ma_20,brent_ma_60,brent_vol_5d,high_low_range,crude_stock_level,gas_stock_level,dist_stock_level,crude_stock_wow_change,gas_stock_wow_change,dist_stock_wow_change,crude_stock_vs_5yr_avg,gas_stock_vs_5yr_avg,dist_stock_vs_5yr_avg,prod_weekly,prod_4w_ma,prod_wow_change,crude_exports,crude_imports,import_4w_ma,net_imports
0,2014-01-02,107.779999,95.440002,12.339996,-0.027256,-0.036819,-0.032930,110.790001,110.6705,109.391334,0.011831,0.033680,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2014-01-03,106.889999,93.959999,12.930000,-0.008258,-0.045455,-0.050879,109.772000,110.3840,109.356167,0.010935,0.017214,326737.0,226959.0,226959.0,-3958.0,6243.0,6243.0,-0.025439,0.048841,0.048841,8145.0,8108.75,0.002955,58.0,7961.0,7680.75,7903.0
2,2014-01-06,106.730003,93.430000,13.300003,-0.001497,-0.048583,-0.046031,108.682001,110.1265,109.310667,0.010182,0.012836,326737.0,226959.0,226959.0,-3958.0,6243.0,6243.0,-0.025439,0.048841,0.048841,8145.0,8108.75,0.002955,58.0,7961.0,7680.75,7903.0
3,2014-01-07,107.349998,93.669998,13.680000,0.005809,-0.034709,-0.032709,107.910001,109.9450,109.271834,0.012422,0.007452,326737.0,226959.0,226959.0,-3958.0,6243.0,6243.0,-0.025439,0.048841,0.048841,8145.0,8108.75,0.002955,58.0,7961.0,7680.75,7903.0
4,2014-01-08,107.150002,92.330002,14.820000,-0.001863,-0.032942,-0.039961,107.180000,109.7220,109.221667,0.012569,0.008493,326737.0,226959.0,226959.0,-3958.0,6243.0,6243.0,-0.025439,0.048841,0.048841,8145.0,8108.75,0.002955,58.0,7961.0,7680.75,7903.0


In [68]:
url = "https://api.eia.gov/v2/seriesid/PET.WPULEUS3.W"
params = {"api_key": API_KEY}

resp = requests.get(url, params=params)
resp.raise_for_status()
js = resp.json()

data = js["response"]["data"]
df = pd.DataFrame(data)
df["period"] = pd.to_datetime(df["period"])
df["refinery_run_rate"] = pd.to_numeric(df["value"])
df = df.sort_values("period").reset_index(drop=True)

ref = df.copy()
ref["period"] = pd.to_datetime(ref["period"])
ref = ref.sort_values("period")

ref_feat = ref[["period", "refinery_run_rate"]].reset_index(drop=True)
print(ref_feat.tail())


         period  refinery_run_rate
1823 2025-10-10               85.7
1824 2025-10-17               88.6
1825 2025-10-24               86.6
1826 2025-10-31               86.0
1827 2025-11-07               89.4


In [69]:
# 형식 통일
ref_feat["period"] = pd.to_datetime(ref_feat["period"])
ref_feat = ref_feat.set_index("period").sort_index()

merged["Date"] = pd.to_datetime(merged["Date"])
merged = merged.set_index("Date").sort_index()

# 병합 (left join + forward fill)
merged = (
    merged
    .merge(ref_feat, how="left", left_index=True, right_index=True)
    .ffill()  # 주간 값은 일단위로 보간
    .reset_index()
    .rename(columns={"index": "Date"})
    .sort_values("Date")
)

merged.tail()

,Date,brent_close,wti_close,brent_wti_spread,brent_ret_1d,brent_ret_5d,brent_ret_20d,brent_ma_5,brent_ma_20,brent_ma_60,brent_vol_5d,high_low_range,crude_stock_level,gas_stock_level,dist_stock_level,crude_stock_wow_change,gas_stock_wow_change,dist_stock_wow_change,crude_stock_vs_5yr_avg,gas_stock_vs_5yr_avg,dist_stock_vs_5yr_avg,prod_weekly,prod_4w_ma,prod_wow_change,crude_exports,crude_imports,import_4w_ma,net_imports,refinery_run_rate
2980,2025-11-10,64.059998,60.130001,3.929996,0.006758,-0.012791,0.011687,63.806001,63.6215,65.813667,0.008451,0.015298,427581.0,205064.0,205064.0,6413.0,-945.0,-945.0,-0.027498,-0.099603,-0.099603,13862.0,13696.5,0.015457,2816.0,5222.0,5528.75,2406.0,89.4
2981,2025-11-11,65.160004,61.040001,4.120003,0.017171,0.011173,0.044398,63.950001,63.7600,65.789667,0.011606,0.025783,427581.0,205064.0,205064.0,6413.0,-945.0,-945.0,-0.027498,-0.099603,-0.099603,13862.0,13696.5,0.015457,2816.0,5222.0,5528.75,2406.0,89.4
2982,2025-11-12,62.709999,58.490002,4.219997,-0.037600,-0.012752,0.012922,63.788000,63.8000,65.738333,0.020894,0.042258,427581.0,205064.0,205064.0,6413.0,-945.0,-945.0,-0.027498,-0.099603,-0.099603,13862.0,13696.5,0.015457,2816.0,5222.0,5528.75,2406.0,89.4
2983,2025-11-13,63.009998,58.689999,4.320000,0.004784,-0.005838,0.031936,63.714000,63.8975,65.674500,0.021141,0.017458,427581.0,205064.0,205064.0,6413.0,-945.0,-945.0,-0.027498,-0.099603,-0.099603,13862.0,13696.5,0.015457,2816.0,5222.0,5528.75,2406.0,89.4
2984,2025-11-14,64.389999,60.090000,4.299999,0.021901,0.011944,0.050579,63.866000,64.0525,65.619833,0.023574,0.027955,427581.0,205064.0,205064.0,6413.0,-945.0,-945.0,-0.027498,-0.099603,-0.099603,13862.0,13696.5,0.015457,2816.0,5222.0,5528.75,2406.0,89.4


# CFTC COT (투기 포지션 / 심리)
- CFTC에서 가져왔는데 brent는 없고 wti만 있음 그래서 얘만 쓸지 brent꺼 크롤링할지 고민해야함.
- 선물시장 참여자들 포지션 변화 통해서, 시장의 투기적 심리랑 헤지 목적, 유동성 측정한 것
- 실제 brent는 ICE에서 거래해서 CFTC에 없는데 WTI로도 글로벌 원유 선물 심리 대용치로 쓴다함.

## 변수들 설명
1. mm_net_long(투기 세력 포지션): 유가 미래 방향에 대한 헤지펀드,CTA 순포지션
-  단기 유가 변동 이끄는 시장 심리를 뜻할 걸요
- ex) 값이 클수록 상승에 배팅을 많이한 상태 -> 강세 심리
2. mm_net_long_ratio(포지션 비율): 전체 시장 중 투기세력 베팅 강도
-  과열, 과매도 구간 탐지 (심리 팽창기 vs 위축)
- ex) 크면 투기세력이 이미 많음 -> 추가 상승 여력 <되돌림 리스크 (계속 그 방향일거 같다)
3. producer_hedge_ratio (생산자 헤지 비율) : 실물생산자의 매도 헤지 규모
- 공급 측 리스트 관리 신호.. 긍까 생산자는 이미 실물을 갖고 있어서 가격하락에 대비하는거임
- ex) 크면 > 다들 리스크 대비중이니까 앞으로 가격이 빠질 수도 있겠다 
4. mm_position_change_wow: 1주동안 포지션 이동 방향
- 유동성 급변/ 리스트 온오프 전환 포착하는것
- ex) 크다 > 새로운 롱 들어옴 > 단기 상승 모멘텀
5. sentiment_position: 순포지션 부호
- 유가 방향성이랑 결합해서 단순 심리 레이블로 
- 그니까 걍 디테일 날리고 지금 시장이 롱인지 숏인지만 보는거임
- +1 > -1 바뀌는 시점만 따로 봐야할 듯

### 걍 정리한 부분
- WTI는 전세계 헤지펀드가 공통적으로 원유 파생상품 포지션 지표로 보니까 Brent 유가 글로벌 심리 대용치 역할한다함.
- 포지션 변화는 뒤에 1-2주간의 가격 움직임에 선행하는 경향 잇음 
- 순매수 전환(+) -> 유사 상승 모멘텀, 순매도로 전환(-) -> 유가 조정중이구나


그러니까 WIT는 '미국의 심리'뿐만 아니라 글로벌 원유 시장의 투기 심리 잘 대변, 그리고 공개/무료라서 안정적으로 쓸 수 있는게 WTI COT임

In [70]:
years = range(2009, 2026)  # 2009~2025까지
base_url = "https://www.cftc.gov/files/dea/history/fut_disagg_txt_{}.zip"

all_df = []

for y in years:
    url = base_url.format(y)
    try:
        print(f"📦 {y}년 다운로드 중...")
        res = requests.get(url)
        res.raise_for_status()
        with zipfile.ZipFile(BytesIO(res.content)) as z:
            for name in z.namelist():
                if name.endswith(".txt"):
                    df = pd.read_csv(z.open(name), header=None)
                    df["year"] = y
                    all_df.append(df)
        print(f"✅ {y} 완료 ({len(all_df)} files total)")
    except Exception as e:
        print(f"❌ {y} 실패: {e}")

df_all = pd.concat(all_df, ignore_index=True)
print("전체 합계:", len(df_all))


📦 2009년 다운로드 중...
❌ 2009 실패: 404 Client Error: Not Found for url: https://www.cftc.gov/files/dea/history/fut_disagg_txt_2009.zip
📦 2010년 다운로드 중...


C:\Users\juhye\AppData\Local\Temp\ipykernel_27616\1113862300.py:15: DtypeWarning: Columns (1,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,131,132,147,148,149,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,188) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(z.open(name), header=None)


✅ 2010 완료 (1 files total)
📦 2011년 다운로드 중...


C:\Users\juhye\AppData\Local\Temp\ipykernel_27616\1113862300.py:15: DtypeWarning: Columns (1,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,131,132,147,148,149,150,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,188) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(z.open(name), header=None)


✅ 2011 완료 (2 files total)
📦 2012년 다운로드 중...


...

✅ 2023 완료 (14 files total)
📦 2024년 다운로드 중...


C:\Users\juhye\AppData\Local\Temp\ipykernel_27616\1113862300.py:15: DtypeWarning: Columns (1,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,131,132,133,134,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,188) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(z.open(name), header=None)


✅ 2024 완료 (15 files total)
📦 2025년 다운로드 중...


C:\Users\juhye\AppData\Local\Temp\ipykernel_27616\1113862300.py:15: DtypeWarning: Columns (1,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,131,132,133,134,145,146,147,148,149,150,153,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,188) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(z.open(name), header=None)


✅ 2025 완료 (16 files total)
전체 합계: 156819


In [71]:
pd.set_option('display.max_columns', None)
df_all.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,year
0,Market_and_Exchange_Names,As_of_Date_In_Form_YYMMDD,Report_Date_as_MM_DD_YYYY,CFTC_Contract_Market_Code,CFTC_Market_Code,CFTC_Region_Code,CFTC_Commodity_Code,Open_Interest_All,Prod_Merc_Positions_Long_All,Prod_Merc_Positions_Short_All,Swap_Positions_Long_All,Swap__Positions_Short_All,Swap__Positions_Spread_All,M_Money_Positions_Long_All,M_Money_Positions_Short_All,M_Money_Positions_Spread_All,Other_Rept_Positions_Long_All,Other_Rept_Positions_Short_All,Other_Rept_Positions_Spread_All,Tot_Rept_Positions_Long_All,Tot_Rept_Positions_Short_All,NonRept_Positions_Long_All,NonRept_Positions_Short_All,Open_Interest_Old,Prod_Merc_Positions_Long_Old,Prod_Merc_Positions_Short_Old,Swap_Positions_Long_Old,Swap__Positions_Short_Old,Swap__Positions_Spread_Old,M_Money_Positions_Long_Old,M_Money_Positions_Short_Old,M_Money_Positions_Spread_Old,Other_Rept_Positions_Long_Old,Other_Rept_Positions_Short_Old,Other_Rept_Positions_Spread_Old,Tot_Rept_Positions_Long_Old,Tot_Rept_Positions_Short_Old,NonRept_Positions_Long_Old,NonRept_Positions_Short_Old,Open_Interest_Other,Prod_Merc_Positions_Long_Other,Prod_Merc_Positions_Short_Other,Swap_Positions_Long_Other,Swap__Positions_Short_Other,Swap__Positions_Spread_Other,M_Money_Positions_Long_Other,M_Money_Positions_Short_Other,M_Money_Positions_Spread_Other,Other_Rept_Positions_Long_Other,Other_Rept_Positions_Short_Other,Other_Rept_Positions_Spread_Other,Tot_Rept_Positions_Long_Other,Tot_Rept_Positions_Short_Other,NonRept_Positions_Long_Other,NonRept_Positions_Short_Other,Change_in_Open_Interest_All,Change_in_Prod_Merc_Long_All,Change_in_Prod_Merc_Short_All,Change_in_Swap_Long_All,Change_in_Swap_Short_All,Change_in_Swap_Spread_All,Change_in_M_Money_Long_All,Change_in_M_Money_Short_All,Change_in_M_Money_Spread_All,Change_in_Other_Rept_Long_All,Change_in_Other_Rept_Short_All,Change_in_Other_Rept_Spread_All,Change_in_Tot_Rept_Long_All,Change_in_Tot_Rept_Short_All,Change_in_NonRept_Long_All,Change_in_NonRept_Short_All,Pct_of_Open_Interest_All,Pct_of_OI_Prod_Merc_Long_All,Pct_of_OI_Prod_Merc_Short_All,Pct_of_OI_Swap_Long_All,Pct_of_OI_Swap_Short_All,Pct_of_OI_Swap_Spread_All,Pct_of_OI_M_Money_Long_All,Pct_of_OI_M_Money_Short_All,Pct_of_OI_M_Money_Spread_All,Pct_of_OI_Other_Rept_Long_All,Pct_of_OI_Other_Rept_Short_All,Pct_of_OI_Other_Rept_Spread_All,Pct_of_OI_Tot_Rept_Long_All,Pct_of_OI_Tot_Rept_Short_All,Pct_of_OI_NonRept_Long_All,Pct_of_OI_NonRept_Short_All,Pct_of_Open_Interest_Old,Pct_of_OI_Prod_Merc_Long_Old,Pct_of_OI_Prod_Merc_Short_Old,Pct_of_OI_Swap_Long_Old,Pct_of_OI_Swap_Short_Old,Pct_of_OI_Swap_Spread_Old,Pct_of_OI_M_Money_Long_Old,Pct_of_OI_M_Money_Short_Old,Pct_of_OI_M_Money_Spread_Old,Pct_of_OI_Other_Rept_Long_Old,Pct_of_OI_Other_Rept_Short_Old,Pct_of_OI_Other_Rept_Spread_Old,Pct_of_OI_Tot_Rept_Long_Old,Pct_of_OI_Tot_Rept_Short_Old,Pct_of_OI_NonRept_Long_Old,Pct_of_OI_NonRept_Short_Old,Pct_of_Open_Interest_Other,Pct_of_OI_Prod_Merc_Long_Other,Pct_of_OI_Prod_Merc_Short_Other,Pct_of_OI_Swap_Long_Other,Pct_of_OI_Swap_Short_Other,Pct_of_OI_Swap_Spread_Other,Pct_of_OI_M_Money_Long_Other,Pct_of_OI_M_Money_Short_Other,Pct_of_OI_M_Money_Spread_Other,Pct_of_OI_Other_Rept_Long_Other,Pct_of_OI_Other_Rept_Short_Other,Pct_of_OI_Other_Rept_Spread_Other,Pct_of_OI_Tot_Rept_Long_Other,Pct_of_OI_Tot_Rept_Short_Other,Pct_of_OI_NonRept_Long_Other,Pct_of_OI_NonRept_Shor

In [72]:
def extract_cot_features(df_all):
    """
    df_all: CFTC Disaggregated Futures Only 데이터 (연도 통합본)
    반환: WTI 주간 / 일간 포지션 피처 (날짜 index)
    """

    # 1️⃣ 헤더 정리
    if df_all.iloc[0, 0] == "Market_and_Exchange_Names":
        df_all.columns = df_all.iloc[0]
        df_all = df_all.drop(index=0).reset_index(drop=True)

    # 2️⃣ 컬럼명 정리
    df_all.columns = [str(c).strip() for c in df_all.columns]
    df_all = df_all.rename(columns=lambda x: x.replace("-", "_").replace(" ", "_"))

    # 날짜 컬럼 자동 탐색
    date_col = next((c for c in df_all.columns if "report_date" in c.lower()), None)
    if not date_col:
        raise KeyError("날짜 컬럼을 찾을 수 없습니다. ('Report_Date_as_' 로 시작해야 함)")

    df_all["date"] = pd.to_datetime(df_all[date_col], errors="coerce")

    # 주요 컬럼 정의
    mm_long = "M_Money_Positions_Long_All"
    mm_short = "M_Money_Positions_Short_All"
    prod_short = "Prod_Merc_Positions_Short_All"
    oi_col = "Open_Interest_All"

    for col in [mm_long, mm_short, prod_short, oi_col]:
        df_all[col] = pd.to_numeric(df_all[col], errors="coerce")

    # 3️⃣ 시장명 소문자로 정리
    df_all["Market_and_Exchange_Names"] = (
        df_all["Market_and_Exchange_Names"].astype(str).str.lower()
    )

    # 4️⃣ 상품 필터링 (WTI만 사용)
    wti = df_all[
        df_all["Market_and_Exchange_Names"].str.contains("crude oil, light sweet", na=False)
    ].copy()

    def make_features(df, prefix):
        df = (
            df.sort_values("date")
              .groupby("date", as_index=False)
              .last()
              .copy()
        )
        df[f"{prefix}_mm_net_long"] = df[mm_long] - df[mm_short]
        df[f"{prefix}_mm_net_long_ratio"] = df[f"{prefix}_mm_net_long"] / df[oi_col]
        df[f"{prefix}_producer_hedge_ratio"] = df[prod_short] / df[oi_col]
        df[f"{prefix}_mm_position_change_wow"] = df[f"{prefix}_mm_net_long"].diff()
        df[f"{prefix}_sentiment_position"] = np.sign(df[f"{prefix}_mm_net_long"])

        # NaN이어도 컬럼 유지
        for col in [
            f"{prefix}_mm_net_long",
            f"{prefix}_mm_net_long_ratio",
            f"{prefix}_producer_hedge_ratio",
            f"{prefix}_mm_position_change_wow",
            f"{prefix}_sentiment_position",
        ]:
            if col not in df.columns:
                df[col] = np.nan

        return df.set_index("date")[
            [
                f"{prefix}_mm_net_long",
                f"{prefix}_mm_net_long_ratio",
                f"{prefix}_producer_hedge_ratio",
                f"{prefix}_mm_position_change_wow",
                f"{prefix}_sentiment_position",
            ]
        ]

    # 5️⃣ 피처 계산 (WTI만)
    wti_feat = make_features(wti, "wti")

    # 6️⃣ 주간 데이터
    cot_weekly = wti_feat.copy()

    # 7️⃣ 일간 데이터로 변환
    cot_daily = cot_weekly.resample("D").ffill().copy()

    return cot_weekly, cot_daily


cot_weekly, cot_daily = extract_cot_features(df_all)

print(cot_weekly.tail())
print(cot_daily.tail())


C:\Users\juhye\AppData\Local\Temp\ipykernel_27616\382429375.py:44: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df.sort_values("date")


            wti_mm_net_long  wti_mm_net_long_ratio  wti_producer_hedge_ratio  \
date                                                                           
2025-08-26         -40862.0              -0.051040                  0.521388   
2025-09-02         -40779.0              -0.051032                  0.529336   
2025-09-09         -40586.0              -0.050581                  0.520728   
2025-09-16         -35887.0              -0.044208                  0.513560   
2025-09-23         -34194.0              -0.042331                  0.509122   

            wti_mm_position_change_wow  wti_sentiment_position  
date                                                            
2025-08-26                      3595.0                    -1.0  
2025-09-02                        83.0                    -1.0  
2025-09-09                       193.0                    -1.0  
2025-09-16                      4699.0                    -1.0  
2025-09-23                      1693.0           

In [73]:
# brent 관련 컬럼 전체 제거
# cot_weekly = cot_weekly.drop(columns=[c for c in cot_weekly.columns if c.startswith("brent_")], errors="ignore")
# cot_daily  = cot_daily.drop(columns=[c for c in cot_daily.columns if c.startswith("brent_")], errors="ignore")
cot_weekly = cot_weekly.reset_index()
cot_daily = cot_daily.reset_index()

In [74]:
cot_weekly = cot_weekly[cot_weekly['date'] >= "2014-01-01"]
cot_daily = cot_daily[cot_daily['date'] >= "2014-01-01"]

In [75]:
cot_weekly.head()

,date,wti_mm_net_long,wti_mm_net_long_ratio,wti_producer_hedge_ratio,wti_mm_position_change_wow,wti_sentiment_position
209,2014-01-07,50512.0,0.093630,0.252797,-5451.0,1.0
210,2014-01-14,51099.0,0.095954,0.236063,587.0,1.0
211,2014-01-21,53128.0,0.100615,0.224717,2029.0,1.0
212,2014-01-28,58400.0,0.105601,0.228352,5272.0,1.0
213,2014-02-04,269712.0,0.172762,0.172313,211312.0,1.0


In [76]:
cot_daily.head()

,date,wti_mm_net_long,wti_mm_net_long_ratio,wti_producer_hedge_ratio,wti_mm_position_change_wow,wti_sentiment_position
1457,2014-01-01,55963.0,0.107468,0.260917,-3848.0,1.0
1458,2014-01-02,55963.0,0.107468,0.260917,-3848.0,1.0
1459,2014-01-03,55963.0,0.107468,0.260917,-3848.0,1.0
1460,2014-01-04,55963.0,0.107468,0.260917,-3848.0,1.0
1461,2014-01-05,55963.0,0.107468,0.260917,-3848.0,1.0


In [77]:
dd=merged

In [78]:
merged = merged.merge(cot_weekly, how="left", left_on="Date", right_on="date")
merged = merged.drop(columns=["date"])
print(merged.head())
print(merged.tail())

        Date  brent_close  wti_close  brent_wti_spread  brent_ret_1d  \
0 2014-01-02   107.779999  95.440002         12.339996     -0.027256   
1 2014-01-03   106.889999  93.959999         12.930000     -0.008258   
2 2014-01-06   106.730003  93.430000         13.300003     -0.001497   
3 2014-01-07   107.349998  93.669998         13.680000      0.005809   
4 2014-01-08   107.150002  92.330002         14.820000     -0.001863   

   brent_ret_5d  brent_ret_20d  brent_ma_5  brent_ma_20  brent_ma_60  \
0     -0.036819      -0.032930  110.790001     110.6705   109.391334   
1     -0.045455      -0.050879  109.772000     110.3840   109.356167   
2     -0.048583      -0.046031  108.682001     110.1265   109.310667   
3     -0.034709      -0.032709  107.910001     109.9450   109.271834   
4     -0.032942      -0.039961  107.180000     109.7220   109.221667   

   brent_vol_5d  high_low_range  crude_stock_level  gas_stock_level  \
0      0.011831        0.033680                NaN             

In [79]:
print(merged.tail())

           Date  brent_close  wti_close  brent_wti_spread  brent_ret_1d  \
2980 2025-11-10    64.059998  60.130001          3.929996      0.006758   
2981 2025-11-11    65.160004  61.040001          4.120003      0.017171   
2982 2025-11-12    62.709999  58.490002          4.219997     -0.037600   
2983 2025-11-13    63.009998  58.689999          4.320000      0.004784   
2984 2025-11-14    64.389999  60.090000          4.299999      0.021901   

      brent_ret_5d  brent_ret_20d  brent_ma_5  brent_ma_20  brent_ma_60  \
2980     -0.012791       0.011687   63.806001      63.6215    65.813667   
2981      0.011173       0.044398   63.950001      63.7600    65.789667   
2982     -0.012752       0.012922   63.788000      63.8000    65.738333   
2983     -0.005838       0.031936   63.714000      63.8975    65.674500   
2984      0.011944       0.050579   63.866000      64.0525    65.619833   

      brent_vol_5d  high_low_range  crude_stock_level  gas_stock_level  \
2980      0.008451      

# 거시 & 기타 (Macro / Risk)
: Brent 움직임에 간접 영향 주는 것들. 좀 보고서 넣을지 말지 정해야함.

1. 달러 인덱스 (DXY)
- 달러 강세 → 유가 하방 압력 경향 (보통 유가랑 달러랑 역상관 관계, 피처 상관 확인해볼 것)
2. 금리
- 긴축/완화 기조, 경기 우려 정도를 유가와 함께 설명하는 용도.
- 금리 상승(특히 2Y): 긴축 > 수요 둔화 우려 > 유가 부담 이런식임
3. 에너지/정유 섹터 가격 (XLE, VDE, IXC)
- 에너지 관련 주식시장 섹터 지수. 유가 기대와 금융시장 심리를 반영.
- ex1) 에너지 ETF만 미리 튀면 → 유가 상방 모멘텀 시그널로 볼 수 있음.
- ex2) 유가 오르는데 ETF 못 따라오면 → 시장이 지속성에 회의적이라고 해석 가능.
4. BDI (발틱 해운 지수) 아직 안넣음 누가 좀 해주셈
- 실물 경기/물동량 축 — 장기 수요 쪽에서
- ex) 상승 > 해상 수요 견조 > 글로벌 실물 수요 굿 > 유가 중장기 수요 측면 ㄱㅊ
5. 이벤트 / 허리케인 / 지정학 더미 (이지만 아직 안찾아봄)
- 근데 뉴스가 알아서 해주지않을까?

In [80]:
TICKERS = {
    "DXY": "DX-Y.NYB",
    "XLE": "XLE",
    "VDE": "VDE",
    "IXC": "IXC",
}

START = "2014-01-01"
END = None  # 오늘까지


def fetch_yf_prices(ticker_map: dict,
                    start: str = "2014-01-01",
                    end: str = None,
                    interval: str = "1d") -> pd.DataFrame:
    """
    여러 티커를 한 번에 받아와서
    종가(Close)만 컬럼으로 가지는 wide-form DataFrame으로 반환.
    """
    all_df = []

    for name, symbol in ticker_map.items():
        df = yf.download(symbol, start=start, end=end,
                         interval=interval, progress=False)

        if df.empty:
            print(f"[WARN] {name} ({symbol}) 데이터 없음")
            continue

        # yfinance 최근 버전: auto_adjust=True 기본 → 'Adj Close' 없을 수 있음
        price_col = "Adj Close" if "Adj Close" in df.columns else "Close"

        out = (
            df[[price_col]]
            .rename(columns={price_col: name})
            .assign(ticker=name)
        )

        all_df.append(out[[name]])

    if not all_df:
        raise RuntimeError("다운로드된 데이터가 없습니다. 티커/네트워크를 확인하세요.")

    # index(Date) 기준으로 병합
    prices = pd.concat(all_df, axis=1)
    prices.index.name = "date"
    return prices



# dxy_ret_1d, xle_ret_1d, vde_ret_1d, ixc_ret_1d : 일간 수익률
def add_basic_features(prices: pd.DataFrame) -> pd.DataFrame:
    """
    DXY, XLE 등에 대해 1일 수익률 등 기본 피처 생성.
    Brent 기준 예측용 테이블로 사용하기 좋게 column명 정리.
    """
    df = prices.copy()

    # 1일 수익률
    for col in df.columns:
        df[f"{col}_ret_1d"] = df[col].pct_change()

    # Brent 타겟 (예: 1일 후 수익률 예측용)
    if "Brent" in df.columns:
        df["Brent_ret_1d_fwd"] = df["Brent"].pct_change().shift(-1)

    return df

In [81]:
prices = fetch_yf_prices(TICKERS, start=START, end=END, interval="1d")
feat_df = add_basic_features(prices)
feat_df.head()

C:\Users\juhye\AppData\Local\Temp\ipykernel_27616\3090514834.py:23: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(symbol, start=start, end=end,
C:\Users\juhye\AppData\Local\Temp\ipykernel_27616\3090514834.py:23: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(symbol, start=start, end=end,
C:\Users\juhye\AppData\Local\Temp\ipykernel_27616\3090514834.py:23: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(symbol, start=start, end=end,
C:\Users\juhye\AppData\Local\Temp\ipykernel_27616\3090514834.py:23: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(symbol, start=start, end=end,
C:\Users\juhye\AppData\Local\Temp\ipykernel_27616\3090514834.py:61: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading 

Price,DXY,XLE,VDE,IXC,"('DXY', 'DX-Y.NYB')_ret_1d","('XLE', 'XLE')_ret_1d","('VDE', 'VDE')_ret_1d","('IXC', 'IXC')_ret_1d"
Ticker,DX-Y.NYB,XLE,VDE,IXC,,,,
date,,,,,,,,
2014-01-02,80.629997,55.728363,85.919952,27.177942,NaN,NaN,NaN,NaN
2014-01-03,80.790001,55.523983,85.713120,27.133377,0.001984,-0.003667,-0.002407,-0.001640
2014-01-06,80.650002,55.600620,85.692421,27.031519,-0.001733,0.001380,-0.000241,-0.003754
2014-01-07,80.830002,56.022167,86.361229,27.241604,0.002232,0.007582,0.007805,0.007772
2014-01-08,81.040001,55.632557,85.802765,27.095188,0.002598,-0.006955,-0.006467,-0.005375


In [82]:
# 멀티인덱스 → 단일 문자열 칼럼으로 변환
feat_df.columns = ['_'.join([str(c) for c in col if c]) for col in feat_df.columns]

# 예: ('Brent', 'BZ=F')_ret_1d → Brent_ret_1d
feat_df = feat_df.rename(columns={
    "Price_XLE": "XLE",
    "Price_VDE": "VDE",
    "Price_IXC": "IXC",
})
feat_df = feat_df.reset_index()
feat_df.head()


,date,DXY_DX-Y.NYB,XLE_XLE,VDE_VDE,IXC_IXC,"('DXY', 'DX-Y.NYB')_ret_1d","('XLE', 'XLE')_ret_1d","('VDE', 'VDE')_ret_1d","('IXC', 'IXC')_ret_1d"
0,2014-01-02,80.629997,55.728363,85.919952,27.177942,NaN,NaN,NaN,NaN
1,2014-01-03,80.790001,55.523983,85.713120,27.133377,0.001984,-0.003667,-0.002407,-0.001640
2,2014-01-06,80.650002,55.600620,85.692421,27.031519,-0.001733,0.001380,-0.000241,-0.003754
3,2014-01-07,80.830002,56.022167,86.361229,27.241604,0.002232,0.007582,0.007805,0.007772
4,2014-01-08,81.040001,55.632557,85.802765,27.095188,0.002598,-0.006955,-0.006467,-0.005375


In [83]:
merged = merged.merge(feat_df, how="left", left_on="Date", right_on="date")
merged = merged.drop(columns=["date"])
merged.tail()

,Date,brent_close,wti_close,brent_wti_spread,brent_ret_1d,brent_ret_5d,brent_ret_20d,brent_ma_5,brent_ma_20,brent_ma_60,brent_vol_5d,high_low_range,crude_stock_level,gas_stock_level,dist_stock_level,crude_stock_wow_change,gas_stock_wow_change,dist_stock_wow_change,crude_stock_vs_5yr_avg,gas_stock_vs_5yr_avg,dist_stock_vs_5yr_avg,prod_weekly,prod_4w_ma,prod_wow_change,crude_exports,crude_imports,import_4w_ma,net_imports,refinery_run_rate,wti_mm_net_long,wti_mm_net_long_ratio,wti_producer_hedge_ratio,wti_mm_position_change_wow,wti_sentiment_position,DXY_DX-Y.NYB,XLE_XLE,VDE_VDE,IXC_IXC,"('DXY', 'DX-Y.NYB')_ret_1d","('XLE', 'XLE')_ret_1d","('VDE', 'VDE')_ret_1d","('IXC', 'IXC')_ret_1d"
2980,2025-11-10,64.059998,60.130001,3.929996,0.006758,-0.012791,0.011687,63.806001,63.6215,65.813667,0.008451,0.015298,427581.0,205064.0,205064.0,6413.0,-945.0,-945.0,-0.027498,-0.099603,-0.099603,13862.0,13696.5,0.015457,2816.0,5222.0,5528.75,2406.0,89.4,NaN,NaN,NaN,NaN,NaN,99.620003,90.349998,127.370003,42.849998,0.000201,0.009046,0.009591,0.009661
2981,2025-11-11,65.160004,61.040001,4.120003,0.017171,0.011173,0.044398,63.950001,63.7600,65.789667,0.011606,0.025783,427581.0,205064.0,205064.0,6413.0,-945.0,-945.0,-0.027498,-0.099603,-0.099603,13862.0,13696.5,0.015457,2816.0,5222.0,5528.75,2406.0,89.4,NaN,NaN,NaN,NaN,NaN,99.459999,91.529999,129.000000,43.400002,-0.001606,0.013060,0.012797,0.012836
2982,2025-11-12,62.709999,58.490002,4.219997,-0.037600,-0.012752,0.012922,63.788000,63.8000,65.738333,0.020894,0.042258,427581.0,205064.0,205064.0,6413.0,-945.0,-945.0,-0.027498,-0.099603,-0.099603,13862.0,13696.5,0.015457,2816.0,5222.0,5528.75,2406.0,89.4,NaN,NaN,NaN,NaN,NaN,99.480003,90.250000,127.190002,42.959999,0.000201,-0.013984,-0.014031,-0.010138
2983,2025-11-13,63.009998,58.689999,4.320000,0.004784,-0.005838,0.031936,63.714000,63.8975,65.674500,0.021141,0.017458,427581.0,205064.0,205064.0,6413.0,-945.0,-945.0,-0.027498,-0.099603,-0.099603,13862.0,13696.5,0.015457,2816.0,5222.0,5528.75,2406.0,89.4,NaN,NaN,NaN,NaN,NaN,99.180000,90.480003,127.279999,42.959999,-0.003016,0.002549,0.000708,0.000000
2984,2025-11-14,64.389999,60.090000,4.299999,0.021901,0.011944,0.050579,63.866000,64.0525,65.619833,0.023574,0.027955,427581.0,205064.0,205064.0,6413.0,-945.0,-945.0,-0.027498,-0.099603,-0.099603,13862.0,13696.5,0.015457,2816.0,5222.0,5528.75,2406.0,89.4,NaN,NaN,NaN,NaN,NaN,99.273003,92.019997,129.339996,43.560001,0.000938,0.017020,0.016185,0.013967


In [84]:
# 금리
# DGS10: 10년 만기 미 국채 수익률
# DGS2: 2년 만기 미 국채 수익률
# spred = 10-2: 긴축/완화 혹은 경기우려 정도 반영

from fredapi import Fred
fred = Fred(api_key='fb8a5fd4f9e0d2bdd044c6b60ed0f2c0')

df10 = fred.get_series('DGS10', start='2013-12-01')
df2  = fred.get_series('DGS2',  start='2013-12-01')

df = pd.DataFrame({'us10y_yield': df10, 'us2y_yield': df2})
df['term_spread'] = df['us10y_yield'] - df['us2y_yield']
df.index.name = 'date'

df = pd.DataFrame({'us10y_yield': df10, 'us2y_yield': df2})
df['term_spread'] = df['us10y_yield'] - df['us2y_yield']
df.index.name = 'date'

df = df.loc['2014-01-01':]  

In [85]:
df = df.reset_index()

df.head()

,date,us10y_yield,us2y_yield,term_spread
0,2014-01-01,NaN,NaN,NaN
1,2014-01-02,3.00,0.39,2.61
2,2014-01-03,3.01,0.41,2.60
3,2014-01-06,2.98,0.40,2.58
4,2014-01-07,2.96,0.40,2.56


In [86]:
merged = merged.merge(df, how="left", left_on="Date", right_on="date")
merged = merged.drop(columns=["date"])
merged.tail()

,Date,brent_close,wti_close,brent_wti_spread,brent_ret_1d,brent_ret_5d,brent_ret_20d,brent_ma_5,brent_ma_20,brent_ma_60,brent_vol_5d,high_low_range,crude_stock_level,gas_stock_level,dist_stock_level,crude_stock_wow_change,gas_stock_wow_change,dist_stock_wow_change,crude_stock_vs_5yr_avg,gas_stock_vs_5yr_avg,dist_stock_vs_5yr_avg,prod_weekly,prod_4w_ma,prod_wow_change,crude_exports,crude_imports,import_4w_ma,net_imports,refinery_run_rate,wti_mm_net_long,wti_mm_net_long_ratio,wti_producer_hedge_ratio,wti_mm_position_change_wow,wti_sentiment_position,DXY_DX-Y.NYB,XLE_XLE,VDE_VDE,IXC_IXC,"('DXY', 'DX-Y.NYB')_ret_1d","('XLE', 'XLE')_ret_1d","('VDE', 'VDE')_ret_1d","('IXC', 'IXC')_ret_1d",us10y_yield,us2y_yield,term_spread
2980,2025-11-10,64.059998,60.130001,3.929996,0.006758,-0.012791,0.011687,63.806001,63.6215,65.813667,0.008451,0.015298,427581.0,205064.0,205064.0,6413.0,-945.0,-945.0,-0.027498,-0.099603,-0.099603,13862.0,13696.5,0.015457,2816.0,5222.0,5528.75,2406.0,89.4,NaN,NaN,NaN,NaN,NaN,99.620003,90.349998,127.370003,42.849998,0.000201,0.009046,0.009591,0.009661,4.13,3.58,0.55
2981,2025-11-11,65.160004,61.040001,4.120003,0.017171,0.011173,0.044398,63.950001,63.7600,65.789667,0.011606,0.025783,427581.0,205064.0,205064.0,6413.0,-945.0,-945.0,-0.027498,-0.099603,-0.099603,13862.0,13696.5,0.015457,2816.0,5222.0,5528.75,2406.0,89.4,NaN,NaN,NaN,NaN,NaN,99.459999,91.529999,129.000000,43.400002,-0.001606,0.013060,0.012797,0.012836,NaN,NaN,NaN
2982,2025-11-12,62.709999,58.490002,4.219997,-0.037600,-0.012752,0.012922,63.788000,63.8000,65.738333,0.020894,0.042258,427581.0,205064.0,205064.0,6413.0,-945.0,-945.0,-0.027498,-0.099603,-0.099603,13862.0,13696.5,0.015457,2816.0,5222.0,5528.75,2406.0,89.4,NaN,NaN,NaN,NaN,NaN,99.480003,90.250000,127.190002,42.959999,0.000201,-0.013984,-0.014031,-0.010138,4.08,3.56,0.52
2983,2025-11-13,63.009998,58.689999,4.320000,0.004784,-0.005838,0.031936,63.714000,63.8975,65.674500,0.021141,0.017458,427581.0,205064.0,205064.0,6413.0,-945.0,-945.0,-0.027498,-0.099603,-0.099603,13862.0,13696.5,0.015457,2816.0,5222.0,5528.75,2406.0,89.4,NaN,NaN,NaN,NaN,NaN,99.180000,90.480003,127.279999,42.959999,-0.003016,0.002549,0.000708,0.000000,4.11,3.58,0.53
2984,2025-11-14,64.389999,60.090000,4.299999,0.021901,0.011944,0.050579,63.866000,64.0525,65.619833,0.023574,0.027955,427581.0,205064.0,205064.0,6413.0,-945.0,-945.0,-0.027498,-0.099603,-0.099603,13862.0,13696.5,0.015457,2816.0,5222.0,5528.75,2406.0,89.4,NaN,NaN,NaN,NaN,NaN,99.273003,92.019997,129.339996,43.560001,0.000938,0.017020,0.016185,0.013967,NaN,NaN,NaN


In [87]:
import pandas as pd
import glob
import os

# 1️⃣ 폴더 내 CSV 모두 합치기
folder = r"..\data\Baltic Dry Index Historical Data"  # 실제 경로로 수정
files = glob.glob(os.path.join(folder, "*.csv"))

dfs = []
for file in files:
    df = pd.read_csv(file)
    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    dfs.append(df)

bdi = pd.concat(dfs, ignore_index=True)
bdi = bdi.sort_values("date").drop_duplicates("date").reset_index(drop=True)

# 2️⃣ 컬럼명에 prefix 추가
bdi = bdi.rename(columns={c: f"bdi_{c}" if c != "date" else "date" for c in bdi.columns})

# 문자열(예: "1,945.00") → 숫자 변환
for col in bdi.columns:
    if col.startswith("bdi_"):
        bdi[col] = (
            bdi[col]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.replace("%", "", regex=False)
        )
        bdi[col] = pd.to_numeric(bdi[col], errors="coerce")

print("✅ BDI 합치기 완료:", bdi.shape)
print(bdi.head())

# 3️⃣ merged랑 날짜 기준 병합
# merged가 period 컬럼(주간) 기준이라면, 일간 데이터를 그대로 붙일 수 있습니다.
merged = merged.merge(bdi, how="left", left_on="Date", right_on="date")

# 중복된 date 컬럼 정리
merged = merged.drop(columns=["date"])

print("✅ merged + bdi 병합 완료:", merged.shape)
print(merged.tail())

✅ BDI 합치기 완료: (2957, 7)
        date  bdi_price  bdi_open  bdi_high  bdi_low  bdi_vol.  bdi_change_%
0 2014-01-02     2113.0    2113.0    2113.0   2113.0       NaN         -7.20
1 2014-01-03     2036.0    2036.0    2036.0   2036.0       NaN         -3.64
2 2014-01-06     1951.0    1951.0    1951.0   1951.0       NaN         -4.17
3 2014-01-07     1876.0    1876.0    1876.0   1876.0       NaN         -3.84
4 2014-01-08     1826.0    1826.0    1826.0   1826.0       NaN         -2.67
✅ merged + bdi 병합 완료: (2985, 51)
           Date  brent_close  wti_close  brent_wti_spread  brent_ret_1d  \
2980 2025-11-10    64.059998  60.130001          3.929996      0.006758   
2981 2025-11-11    65.160004  61.040001          4.120003      0.017171   
2982 2025-11-12    62.709999  58.490002          4.219997     -0.037600   
2983 2025-11-13    63.009998  58.689999          4.320000      0.004784   
2984 2025-11-14    64.389999  60.090000          4.299999      0.021901   

      brent_ret_5d  brent_ret_

In [88]:
merged.drop(columns=["bdi_vol."], inplace=True)

In [89]:
merged.head()

,Date,brent_close,wti_close,brent_wti_spread,brent_ret_1d,brent_ret_5d,brent_ret_20d,brent_ma_5,brent_ma_20,brent_ma_60,brent_vol_5d,high_low_range,crude_stock_level,gas_stock_level,dist_stock_level,crude_stock_wow_change,gas_stock_wow_change,dist_stock_wow_change,crude_stock_vs_5yr_avg,gas_stock_vs_5yr_avg,dist_stock_vs_5yr_avg,prod_weekly,prod_4w_ma,prod_wow_change,crude_exports,crude_imports,import_4w_ma,net_imports,refinery_run_rate,wti_mm_net_long,wti_mm_net_long_ratio,wti_producer_hedge_ratio,wti_mm_position_change_wow,wti_sentiment_position,DXY_DX-Y.NYB,XLE_XLE,VDE_VDE,IXC_IXC,"('DXY', 'DX-Y.NYB')_ret_1d","('XLE', 'XLE')_ret_1d","('VDE', 'VDE')_ret_1d","('IXC', 'IXC')_ret_1d",us10y_yield,us2y_yield,term_spread,bdi_price,bdi_open,bdi_high,bdi_low,bdi_change_%
0,2014-01-02,107.779999,95.440002,12.339996,-0.027256,-0.036819,-0.032930,110.790001,110.6705,109.391334,0.011831,0.033680,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,80.629997,55.728363,85.919952,27.177942,NaN,NaN,NaN,NaN,3.00,0.39,2.61,2113.0,2113.0,2113.0,2113.0,-7.20
1,2014-01-03,106.889999,93.959999,12.930000,-0.008258,-0.045455,-0.050879,109.772000,110.3840,109.356167,0.010935,0.017214,326737.0,226959.0,226959.0,-3958.0,6243.0,6243.0,-0.025439,0.048841,0.048841,8145.0,8108.75,0.002955,58.0,7961.0,7680.75,7903.0,92.3,NaN,NaN,NaN,NaN,NaN,80.790001,55.523983,85.713120,27.133377,0.001984,-0.003667,-0.002407,-0.001640,3.01,0.41,2.60,2036.0,2036.0,2036.0,2036.0,-3.64
2,2014-01-06,106.730003,93.430000,13.300003,-0.001497,-0.048583,-0.046031,108.682001,110.1265,109.310667,0.010182,0.012836,326737.0,226959.0,226959.0,-3958.0,6243.0,6243.0,-0.025439,0.048841,0.048841,8145.0,8108.75,0.002955,58.0,7961.0,7680.75,7903.0,92.3,NaN,NaN,NaN,NaN,NaN,80.650002,55.600620,85.692421,27.031519,-0.001733,0.001380,-0.000241,-0.003754,2.98,0.40,2.58,1951.0,1951.0,1951.0,1951.0,-4.17
3,2014-01-07,107.349998,93.669998,13.680000,0.005809,-0.034709,-0.032709,107.910001,109.9450,109.271834,0.012422,0.007452,326737.0,226959.0,226959.0,-3958.0,6243.0,6243.0,-0.025439,0.048841,0.048841,8145.0,8108.75,0.002955,58.0,7961.0,7680.75,7903.0,92.3,50512.0,0.09363,0.252797,-5451.0,1.0,80.830002,56.022167,86.361229,27.241604,0.002232,0.007582,0.007805,0.007772,2.96,0.40,2.56,1876.0,1876.0,1876.0,1876.0,-3.84
4,2014-01-08,107.150002,92.330002,14.820000,-0.001863,-0.032942,-0.039961,107.180000,109.7220,109.221667,0.012569,0.008493,326737.0,226959.0,226959.0,-3958.0,6243.0,6243.0,-0.025439,0.048841,0.048841,8145.0,8108.75,0.002955,58.0,7961.0,7680.75,7903.0,92.3,NaN,NaN,NaN,NaN,NaN,81.040001,55.632557,85.802765,27.095188,0.002598,-0.006955,-0.006467,-0.005375,3.01,0.43,2.58,1826.0,1826.0,1826.0,1826.0,-2.67


In [90]:
merged.tail()

,Date,brent_close,wti_close,brent_wti_spread,brent_ret_1d,brent_ret_5d,brent_ret_20d,brent_ma_5,brent_ma_20,brent_ma_60,brent_vol_5d,high_low_range,crude_stock_level,gas_stock_level,dist_stock_level,crude_stock_wow_change,gas_stock_wow_change,dist_stock_wow_change,crude_stock_vs_5yr_avg,gas_stock_vs_5yr_avg,dist_stock_vs_5yr_avg,prod_weekly,prod_4w_ma,prod_wow_change,crude_exports,crude_imports,import_4w_ma,net_imports,refinery_run_rate,wti_mm_net_long,wti_mm_net_long_ratio,wti_producer_hedge_ratio,wti_mm_position_change_wow,wti_sentiment_position,DXY_DX-Y.NYB,XLE_XLE,VDE_VDE,IXC_IXC,"('DXY', 'DX-Y.NYB')_ret_1d","('XLE', 'XLE')_ret_1d","('VDE', 'VDE')_ret_1d","('IXC', 'IXC')_ret_1d",us10y_yield,us2y_yield,term_spread,bdi_price,bdi_open,bdi_high,bdi_low,bdi_change_%
2980,2025-11-10,64.059998,60.130001,3.929996,0.006758,-0.012791,0.011687,63.806001,63.6215,65.813667,0.008451,0.015298,427581.0,205064.0,205064.0,6413.0,-945.0,-945.0,-0.027498,-0.099603,-0.099603,13862.0,13696.5,0.015457,2816.0,5222.0,5528.75,2406.0,89.4,NaN,NaN,NaN,NaN,NaN,99.620003,90.349998,127.370003,42.849998,0.000201,0.009046,0.009591,0.009661,4.13,3.58,0.55,NaN,NaN,NaN,NaN,NaN
2981,2025-11-11,65.160004,61.040001,4.120003,0.017171,0.011173,0.044398,63.950001,63.7600,65.789667,0.011606,0.025783,427581.0,205064.0,205064.0,6413.0,-945.0,-945.0,-0.027498,-0.099603,-0.099603,13862.0,13696.5,0.015457,2816.0,5222.0,5528.75,2406.0,89.4,NaN,NaN,NaN,NaN,NaN,99.459999,91.529999,129.000000,43.400002,-0.001606,0.013060,0.012797,0.012836,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2982,2025-11-12,62.709999,58.490002,4.219997,-0.037600,-0.012752,0.012922,63.788000,63.8000,65.738333,0.020894,0.042258,427581.0,205064.0,205064.0,6413.0,-945.0,-945.0,-0.027498,-0.099603,-0.099603,13862.0,13696.5,0.015457,2816.0,5222.0,5528.75,2406.0,89.4,NaN,NaN,NaN,NaN,NaN,99.480003,90.250000,127.190002,42.959999,0.000201,-0.013984,-0.014031,-0.010138,4.08,3.56,0.52,NaN,NaN,NaN,NaN,NaN
2983,2025-11-13,63.009998,58.689999,4.320000,0.004784,-0.005838,0.031936,63.714000,63.8975,65.674500,0.021141,0.017458,427581.0,205064.0,205064.0,6413.0,-945.0,-945.0,-0.027498,-0.099603,-0.099603,13862.0,13696.5,0.015457,2816.0,5222.0,5528.75,2406.0,89.4,NaN,NaN,NaN,NaN,NaN,99.180000,90.480003,127.279999,42.959999,-0.003016,0.002549,0.000708,0.000000,4.11,3.58,0.53,NaN,NaN,NaN,NaN,NaN
2984,2025-11-14,64.389999,60.090000,4.299999,0.021901,0.011944,0.050579,63.866000,64.0525,65.619833,0.023574,0.027955,427581.0,205064.0,205064.0,6413.0,-945.0,-945.0,-0.027498,-0.099603,-0.099603,13862.0,13696.5,0.015457,2816.0,5222.0,5528.75,2406.0,89.4,NaN,NaN,NaN,NaN,NaN,99.273003,92.019997,129.339996,43.560001,0.000938,0.017020,0.016185,0.013967,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [91]:
len(merged)

2985

In [92]:
merged.to_csv("data.csv", index=False)